Merging the news data and the yfinance data into one dataset for testing training and visualizing

In [85]:
import pandas as pd

news_data = pd.read_csv("data/news/cleaned/all_clean.csv")

news_data = news_data.sort_values(by="date", ascending=True).reset_index(drop=True)

news_data.head()

,date,summary
0,2017-12-18,france save marquis de sade 120 day sodom auct...
1,2017-12-19,house price fall london southeast 2018 say sur...
2,2017-12-20,hedge fund fail stop billiondollar brain city ...
3,2017-12-21,guardian brexit watch brexit helped push down ...
4,2017-12-22,steelworker face huge pension cut tata complet...


In [86]:
yfinance_data = pd.read_csv("data/stock/yfinance_data_cleaned.csv")

# sort by date ascending
yfinance_data = yfinance_data.sort_values(by="date", ascending=True).reset_index(
    drop=True
)
yfinance_data.head()

,volume,pct_change,target,date
0,3407680000,-0.003230,0,2017-12-19
1,3246230000,-0.000828,1,2017-12-20
2,3293130000,0.001986,0,2017-12-21
3,2401030000,-0.000458,0,2017-12-22
4,1970660000,-0.001058,1,2017-12-26


In [87]:
# Combining datasets based on time outer join so that it keeps all data from both datasets
merged_data = pd.merge(news_data, yfinance_data, on='date', how='outer')

We want to see where we have rows that are missing either finance or news data.

For the news data, we will just drop those

For the finance data, we should shift our news data entries to the next day that has finance data available because that will impact our sentiment analysis 

In [88]:
merged_data[merged_data['summary'].isnull() | (merged_data['summary'] == '')]

,date,summary,volume,pct_change,target
9,2017-12-29,NaN,2.447760e+09,-0.005183,1.0
41,2018-02-13,NaN,3.503540e+09,0.002613,1.0


In [89]:
merged_data = merged_data[merged_data['summary'].notnull()].reset_index(drop=True)

In [90]:
merged_data[merged_data['summary'].isnull() | (merged_data['summary'] == '')]

,date,summary,volume,pct_change,target


Great now lets do the empty finance data rows

In [91]:
merged_data[merged_data["volume"].isnull()]

,date,summary,volume,pct_change,target
0,2017-12-18,france save marquis de sade 120 day sodom auct...,NaN,NaN,NaN
5,2017-12-25,cramer say owning many stock little cash set u...,NaN,NaN,NaN
9,2018-01-01,hammond relying household debt hit target say ...,NaN,NaN,NaN
19,2018-01-15,cramer remix tuesday day buy facebook jim cram...,NaN,NaN,NaN
43,2018-02-19,top stock exchange ceo urge caution over profi...,NaN,NaN,NaN
72,2018-03-30,cramers lightning round cv management no good ...,NaN,NaN,NaN
113,2018-05-28,landlord fight use cva retailer seeking rent c...,NaN,NaN,NaN
140,2018-07-04,world cup hot weather bolster supermarket sale...,NaN,NaN,NaN
183,2018-09-03,energy bill 10m customer find save 100 ten yea...,NaN,NaN,NaN
241,2018-11-22,nil pratley finance british gas unable mask cu...,NaN,NaN,NaN


comment stuff here


In [92]:
import numpy as np

# 2. Create a "grouper" column
# We identify valid trading days. If volume exists, we keep the date.
# If volume is NaN/0, we set it to NaT (Not a Time) so we can fill it later.
merged_data["trading_day_group"] = np.where(
    merged_data["volume"].notnull(), merged_data["date"], np.nan
)

# 3. Backfill the dates
# This is the magic step. It takes the date of the next valid row
# and pulls it UP into the previous NaN rows.
merged_data["trading_day_group"] = merged_data["trading_day_group"].bfill()

# 4. Group by this new column and aggregate
# We combine the summaries and keep the financial data from the valid trading day (the last entry in the group)
shifted_data = (
    merged_data.groupby("trading_day_group")
    .agg(
        {
            "date": "last",  # Keep the actual trading date
            "summary": " ".join,  # Join the news strings together with a space
            "volume": "last",  # Take the volume from the valid day
            "pct_change": "last",  # Take the change from the valid day
            "target": "last",  # Take the target from the valid day
        }
    )
    .reset_index(drop=True)
)

# Optional: Remove any rows that remained NaN (e.g., if the very last row in your dataset was a holiday)
shifted_data = shifted_data.dropna(subset=["volume"])

In [93]:
shifted_data[shifted_data["volume"].isnull()]

,date,summary,volume,pct_change,target


In [94]:
shifted_data.to_csv('data/merged_data.csv', index=False)
shifted_data.head()

,date,summary,volume,pct_change,target
0,2017-12-19,france save marquis de sade 120 day sodom auct...,3.407680e+09,-0.003230,0.0
1,2017-12-20,hedge fund fail stop billiondollar brain city ...,3.246230e+09,-0.000828,1.0
2,2017-12-21,guardian brexit watch brexit helped push down ...,3.293130e+09,0.001986,0.0
3,2017-12-22,steelworker face huge pension cut tata complet...,2.401030e+09,-0.000458,0.0
4,2017-12-26,cramer say owning many stock little cash set u...,1.970660e+09,-0.001058,1.0
